# Step 1 - Data exploration
Brain MRI classification dataset: one folder per class in `data/classification/raw`.

## Import libraries

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

In [ ]:
DATA_DIR = Path("../data/classification/raw")
ALLOWED_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp"]

## Classes
The class names are the folder names.

In [ ]:
def get_class_names(data_dir):
    class_names = []
    for folder in data_dir.iterdir():
        if folder.is_dir():
            class_names.append(folder.name)
    return sorted(class_names)

In [ ]:
class_names = get_class_names(DATA_DIR)
print(class_names)

## Check the images
Keep only allowed extensions. Use try/except to skip images that cannot be opened.

In [ ]:
def is_valid_image(path):
    if path.suffix.lower() not in ALLOWED_EXTENSIONS:
        return False
    try:
        with Image.open(path) as image:
            image.verify()
        return True
    except Exception as error:
        print("Cannot open", path, "->", error)
        return False

In [ ]:
def load_image_list(data_dir, class_names):
    rows = []
    rejected = []
    for class_name in class_names:
        for path in (data_dir / class_name).iterdir():
            if is_valid_image(path):
                rows.append({"path": str(path), "class": class_name})
            else:
                rejected.append(path)
    return pd.DataFrame(rows), rejected

In [ ]:
df, rejected = load_image_list(DATA_DIR, class_names)
print("Valid images:", len(df))
print("Rejected files:", len(rejected))
df.head()

## Number of images per class

In [ ]:
df["class"].value_counts()

In [ ]:
def plot_class_counts(df):
    plt.figure(figsize=(8, 4))
    sns.countplot(data=df, x="class", order=class_names)
    plt.title("Number of images per class")
    plt.show()

In [ ]:
plot_class_counts(df)

## Some images of each class

In [ ]:
def show_samples(df, class_names, n=5):
    fig, axes = plt.subplots(len(class_names), n, figsize=(n * 2.5, len(class_names) * 2.5))
    for row, class_name in enumerate(class_names):
        paths = df[df["class"] == class_name]["path"].sample(n, random_state=42)
        for col, path in enumerate(paths):
            image = Image.open(path).convert("L")
            axes[row, col].imshow(image, cmap="gray")
            axes[row, col].set_title(class_name, fontsize=9)
            axes[row, col].axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
show_samples(df, class_names)